# Préparation des données pour le Lakehouse Fabric

**Objectif** : transformer les données utilisées jusqu'ici (stations Citiz, contours administratifs, hypothèses de coût) en **tables propres, typées et documentées**, prêtes pour le Lakehouse.

**Organisation du Lakehouse**

```
Files/
├── raw/citiz/     CSV tels que sortis de citiz_gbfs_extraction.ipynb (archive, jamais modifiés)
├── raw/ign/       contours régions / départements (GeoJSON)
└── clean/         une copie Parquet de chaque table (format d'échange)
Tables/            tables Delta, interrogeables en SQL et par Power BI
```

**Fonctionne dans les deux environnements**
- **Dans Fabric** (`spark` existe) : lit `Files/raw/`, écrit `Files/clean/*.parquet` puis les tables Delta.
- **En local** : copie `output/*.csv` dans `lakehouse/Files/raw/citiz/`, et écrit `lakehouse/Files/clean/*.parquet`. Le dossier `lakehouse/` reproduit l'arborescence Fabric : il suffit de le déposer tel quel dans le Lakehouse.

**Règles appliquées à toutes les tables**
- noms de tables et de colonnes en minuscules, sans espace ni accent (contrainte Delta / SQL) ;
- identifiants en texte (`"01"`, `"2A"` ne doivent pas devenir des nombres) ;
- dates en UTC, précision microseconde (Spark ne lit pas les nanosecondes) ;
- géométries en texte WKT (`geometry_wkt`) + `lat` / `lon` : Delta n'a pas de type géométrique, Power BI sait lire lat / lon ;
- une clé primaire par table, vérifiée avant écriture ;
- colonne `_loaded_at` : date du chargement, pour la traçabilité.

In [ ]:
# geopandas n'est pas garanti dans le runtime Fabric : installation pour la session
%pip install -q geopandas

In [ ]:
import json, shutil
from datetime import datetime, timezone
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd

try:
    spark  # noqa: F821  (défini automatiquement dans Fabric)
    IN_FABRIC = True
except NameError:
    IN_FABRIC = False

if IN_FABRIC:
    LAKEHOUSE = Path("/lakehouse/default")          # Lakehouse attaché au notebook
else:
    ROOT = Path("..") if Path("../output").exists() else Path(".")
    LAKEHOUSE = ROOT / "lakehouse"                   # miroir local de l'arborescence Fabric

RAW_CITIZ = LAKEHOUSE / "Files/raw/citiz"
RAW_IGN = LAKEHOUSE / "Files/raw/ign"
CLEAN = LAKEHOUSE / "Files/clean"
for d in (RAW_CITIZ, RAW_IGN, CLEAN):
    d.mkdir(parents=True, exist_ok=True)

LOADED_AT = pd.Timestamp(datetime.now(timezone.utc))
print("Fabric" if IN_FABRIC else "Local", "->", LAKEHOUSE.resolve())

## 1. Rassembler les fichiers bruts dans `Files/raw/`

- **En local** : on copie les CSV de `output/`.
- **Dans Fabric** : déposer au préalable les 4 CSV de `output/` dans `Files/raw/citiz/` (Lakehouse > Files > Upload), ou faire tourner `citiz_gbfs_extraction.ipynb` dans Fabric.

Les contours IGN sont téléchargés directement dans `Files/raw/ign/`.

In [ ]:
CITIZ_FILES = ["citiz_flux.csv", "citiz_stations.csv", "citiz_station_status.csv", "citiz_vehicle_types.csv"]

if not IN_FABRIC:
    for f in CITIZ_FILES:
        shutil.copy(ROOT / "output" / f, RAW_CITIZ / f)

manquants = [f for f in CITIZ_FILES if not (RAW_CITIZ / f).exists()]
assert not manquants, f"À déposer dans Files/raw/citiz/ : {manquants}"

GEOJSON = {
    "regions": "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions-version-simplifiee.geojson",
    "departements": "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson",
}
for name, url in GEOJSON.items():
    path = RAW_IGN / f"{name}.geojson"
    if not path.exists():
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        path.write_bytes(r.content)
print(sorted(p.name for p in RAW_CITIZ.iterdir()), sorted(p.name for p in RAW_IGN.iterdir()))

## 2. Outils communs

- `typed` : garde uniquement les colonnes du schéma, dans l'ordre, avec le bon type. Une colonne absente du fichier source est créée vide, pour que le schéma reste stable d'une exécution à l'autre.
- `save_table` : vérifie la clé primaire, écrit le Parquet dans `Files/clean/`, puis (dans Fabric) la table Delta.
- `read_raw` : lit un CSV brut en forçant les identifiants en texte.

In [ ]:
def typed(df, schema):
    """Projette df sur schema = {colonne: type} ; types : string, Int64, float64, boolean, datetime."""
    out = pd.DataFrame(index=df.index)
    for col, dtype in schema.items():
        s = df[col] if col in df else pd.Series(pd.NA, index=df.index, dtype="object")
        if dtype == "datetime":
            out[col] = pd.to_datetime(s, utc=True, errors="coerce", format="ISO8601")
        else:
            out[col] = s.astype(dtype)
    return out.reset_index(drop=True)

def save_table(df, name, pk):
    """Contrôle la clé primaire puis écrit Files/clean/<name>.parquet (+ table Delta dans Fabric)."""
    dup = df.duplicated(pk).sum()
    assert dup == 0, f"{name} : {dup} doublons sur la clé {pk}"
    assert df[pk].notna().all().all(), f"{name} : clé {pk} incomplète"
    df = df.assign(_loaded_at=LOADED_AT)
    df.to_parquet(CLEAN / f"{name}.parquet", index=False,
                  coerce_timestamps="us", allow_truncated_timestamps=True)
    if IN_FABRIC:
        (spark.read.parquet(f"Files/clean/{name}.parquet")          # noqa: F821
              .write.format("delta").mode("overwrite")
              .option("overwriteSchema", "true").saveAsTable(name))
    print(f"{name:<28} {len(df):>6} lignes  clé={pk}")
    return df

ID_COLS = {"station_id": "string", "vehicle_type_id": "string", "post_code": "string"}

def read_raw(file):
    return pd.read_csv(RAW_CITIZ / file, dtype=ID_COLS)

tables = {}   # nom -> DataFrame écrit, pour les contrôles de fin

## 3. Référentiel géographique : `geo_region`, `geo_departement`

Le département porte le code de sa région (jointure spatiale sur un point intérieur du département), ce qui permet ensuite de filtrer « Rhône » ou « Auvergne-Rhône-Alpes » sans recalcul géographique.

In [ ]:
regions = gpd.read_file(RAW_IGN / "regions.geojson")
departements = gpd.read_file(RAW_IGN / "departements.geojson")

points = departements.copy()
points["geometry"] = departements.representative_point()
dep_region = gpd.sjoin(points, regions.rename(columns={"code": "region_code"})[["region_code", "geometry"]],
                       how="left", predicate="within")
departements["region_code"] = dep_region["region_code"].values

def geo_table(gdf, schema):
    df = pd.DataFrame(gdf.drop(columns="geometry"))
    df["geometry_wkt"] = gdf.geometry.to_wkt()
    c = gdf.to_crs("EPSG:2154").geometry.centroid.to_crs("EPSG:4326")
    df["centroid_lat"], df["centroid_lon"] = c.y, c.x
    return typed(df, schema)

GEO_SCHEMA = {"code": "string", "nom": "string", "centroid_lat": "float64", "centroid_lon": "float64",
              "geometry_wkt": "string"}
tables["geo_region"] = save_table(geo_table(regions, GEO_SCHEMA), "geo_region", ["code"])
tables["geo_departement"] = save_table(
    geo_table(departements, {**GEO_SCHEMA, "region_code": "string"}), "geo_departement", ["code"])

## 4. Citiz : réseaux, stations, types de véhicules

- `citiz_reseau` : un flux GBFS régional par ligne (clé `source`), avec un nom court lisible (`reseau`).
- `citiz_station` : clé `station_uid` = `source:station_id` (un même `station_id` existe dans plusieurs réseaux). On retire ici l'état instantané (`num_vehicles_available`, `is_renting`) : il appartient à la table de statut. On ajoute le département et la région.
- `citiz_vehicle_type` : clé `vehicle_type_uid` = `source:vehicle_type_id`.

In [ ]:
flux = read_raw("citiz_flux.csv")
flux["reseau"] = flux.dataset_title.str.replace("Autopartage Citiz ", "", regex=False)
tables["citiz_reseau"] = save_table(typed(flux, {
    "source": "string", "reseau": "string", "dataset_title": "string", "publisher": "string",
    "gbfs_url": "string", "gbfs_version": "string", "feeds": "string", "licence": "string",
    "page_url": "string", "dataset_id": "string", "is_available": "boolean",
}), "citiz_reseau", ["source"])

In [ ]:
st = read_raw("citiz_stations.csv")
st["station_uid"] = st.source + ":" + st.station_id
gst = gpd.GeoDataFrame(st, geometry=gpd.points_from_xy(st.lon, st.lat), crs="EPSG:4326")
gst = gpd.sjoin(gst, departements[["code", "region_code", "geometry"]].rename(columns={"code": "dep_code"}),
                how="left", predicate="within").drop(columns="index_right")
print("Stations sans département :", gst.dep_code.isna().sum())
gst["geometry_wkt"] = gst.geometry.to_wkt()

tables["citiz_station"] = save_table(typed(gst, {
    "station_uid": "string", "source": "string", "station_id": "string", "name": "string",
    "address": "string", "post_code": "string", "city": "string",
    "lat": "float64", "lon": "float64", "geometry_wkt": "string",
    "dep_code": "string", "region_code": "string",
    "capacity": "Int64", "is_virtual_station": "boolean", "sources": "string",
    "extracted_at": "datetime",
}), "citiz_station", ["station_uid"])

In [ ]:
vt = read_raw("citiz_vehicle_types.csv")
vt["vehicle_type_uid"] = vt.source + ":" + vt.vehicle_type_id
tables["citiz_vehicle_type"] = save_table(typed(vt, {
    "vehicle_type_uid": "string", "source": "string", "vehicle_type_id": "string", "name": "string",
    "make": "string", "model": "string", "form_factor": "string", "propulsion_type": "string",
    "rider_capacity": "Int64", "max_range_meters": "Int64",
}), "citiz_vehicle_type", ["vehicle_type_uid"])

## 5. Citiz : historique de disponibilité

- `citiz_station_status` : une ligne par station et par instantané (clé `station_uid` + `snapshot_ts`). C'est la table qui grossit à chaque extraction et qui servira d'indicateur d'usage.
- `citiz_station_status_vehicle` : la colonne JSON `vehicle_types_available` dépliée en lignes (une par type de véhicule disponible), pour compter par exemple les voitures électriques disponibles sans parser de JSON dans Power BI.

`num_vehicles_disabled` est vide dans tous les flux actuels : on la garde pour la stabilité du schéma.

In [ ]:
status = read_raw("citiz_station_status.csv")
status["station_uid"] = status.source + ":" + status.station_id
status = status.drop_duplicates(["station_uid", "snapshot_ts"], keep="last")

tables["citiz_station_status"] = save_table(typed(status, {
    "station_uid": "string", "snapshot_ts": "datetime", "source": "string", "station_id": "string",
    "num_vehicles_available": "Int64", "num_vehicles_disabled": "Int64", "num_docks_available": "Int64",
    "is_installed": "boolean", "is_renting": "boolean", "is_returning": "boolean",
    "last_reported": "datetime",
}), "citiz_station_status", ["station_uid", "snapshot_ts"])

rows = []
for r in status.itertuples():
    if isinstance(r.vehicle_types_available, str):
        for v in json.loads(r.vehicle_types_available):
            rows.append({"station_uid": r.station_uid, "snapshot_ts": r.snapshot_ts,
                         "vehicle_type_uid": f"{r.source}:{v['vehicle_type_id']}", "count": v.get("count")})
tables["citiz_station_status_vehicle"] = save_table(typed(pd.DataFrame(rows), {
    "station_uid": "string", "snapshot_ts": "datetime", "vehicle_type_uid": "string", "count": "Int64",
}), "citiz_station_status_vehicle", ["station_uid", "snapshot_ts", "vehicle_type_uid"])

## 6. Paramètres de coût : `param_cout`

Table de paramètres pour le modèle de coût de l'objectif 3. Chaque ligne indique si la valeur est **sourcée**, **calculée** ou une **hypothèse** à fixer : c'est ce qu'on montrera au jury comme « limites reconnues ».
Les valeurs vides sont à compléter (idéalement avec les chiffres donnés par Citiz). Modifier cette cellule puis relancer pour mettre la table à jour.

In [ ]:
CONVENTION = "https://genevoisfrancais.org/wp-content/uploads/CS2021_23.2-AUTOPARTAGE_Convention-cadreAnnexe.pdf"
LIVRE_BLANC = "https://www.citiz.coop/uploads/2026/02/2026-CITIZ-livre_blanc.pdf"

couts = pd.DataFrame([
    # equipement, parametre,                        valeur,  unite,               statut,      source
    ("citiz", "boitier_telematique",              4000,    "EUR HT / voiture",  "sourcé",    CONVENTION + " (art. 5.3)"),
    ("citiz", "frais_fixes_exploitation",          250,    "EUR / mois / voiture", "sourcé", CONVENTION + " (art. 6.4)"),
    ("citiz", "recette_par_km",                   0.19,    "EUR / km",          "sourcé",    CONVENTION + " (art. 6.4)"),
    ("citiz", "voitures_min_par_station",            2,    "voitures",          "sourcé",    CONVENTION + " (préambule)"),
    ("citiz", "seuil_rentabilite_km",       250 / 0.19,    "km / mois / voiture", "calculé", "frais_fixes_exploitation / recette_par_km"),
    ("citiz", "prix_achat_voiture",               None,    "EUR TTC / voiture", "hypothèse", "à fixer (demander à Citiz)"),
    ("citiz", "amenagement_station",              None,    "EUR / station",     "hypothèse", LIVRE_BLANC + " : coût qualifié de « minime »"),
    ("arret_bus", "creation_arret",               None,    "EUR / arrêt",       "hypothèse", "à rechercher"),
    ("velo_libre_service", "creation_station",    None,    "EUR / station",     "hypothèse", "à rechercher"),
], columns=["equipement", "parametre", "valeur", "unite", "statut", "source"])

tables["param_cout"] = save_table(typed(couts, {
    "equipement": "string", "parametre": "string", "valeur": "float64",
    "unite": "string", "statut": "string", "source": "string",
}), "param_cout", ["equipement", "parametre"])

## 7. Contrôles de qualité

Vérifie les liens entre tables (clés étrangères) : un statut doit pointer vers une station connue, une station vers un réseau et un département connus, etc.

In [ ]:
def check_fk(child, col, parent, pcol):
    orphelins = ~tables[child][col].dropna().isin(tables[parent][pcol])
    print(f"{'OK ' if orphelins.sum() == 0 else 'KO '} {child}.{col} -> {parent}.{pcol} : {orphelins.sum()} orphelins")

check_fk("citiz_station", "source", "citiz_reseau", "source")
check_fk("citiz_station", "dep_code", "geo_departement", "code")
check_fk("citiz_station_status", "station_uid", "citiz_station", "station_uid")
check_fk("citiz_station_status_vehicle", "vehicle_type_uid", "citiz_vehicle_type", "vehicle_type_uid")
check_fk("citiz_vehicle_type", "source", "citiz_reseau", "source")
check_fk("geo_departement", "region_code", "geo_region", "code")

In [ ]:
# Dictionnaire de données : schéma réel de chaque table écrite
dictionnaire = pd.DataFrame([
    {"table": t, "colonne": c, "type": str(df[c].dtype), "nb_vides": int(df[c].isna().sum()), "lignes": len(df)}
    for t, df in tables.items() for c in df.columns
])
dictionnaire.to_csv(CLEAN / "_dictionnaire.csv", index=False)
dictionnaire.groupby("table").agg(lignes=("lignes", "first"), colonnes=("colonne", "size"))

## 8. Vérification dans Fabric

Requêtes SQL de contrôle sur les tables Delta (Fabric uniquement). Le même SQL marche dans le point de terminaison SQL du Lakehouse.

In [ ]:
if IN_FABRIC:
    display(spark.sql("""
        SELECT d.nom AS departement, COUNT(*) AS stations, SUM(s.capacity) AS capacite
        FROM citiz_station s JOIN geo_departement d ON s.dep_code = d.code
        GROUP BY d.nom ORDER BY stations DESC LIMIT 10
    """))  # noqa: F821
else:
    print("Local : fichiers prêts dans", CLEAN.resolve())
    print("Déposer le contenu de lakehouse/Files/ dans Files/ du Lakehouse, puis relancer ce notebook dans Fabric.")